# LAB 6 &ndash; DSA

As usual, start by writing <span style="color: red">HACKOOLIQUES</span>.

## A) Verification, pt. 1

In the file <i>message.py</i> you will find what I've got to say to you all (the file can be directly interpreted in Python, but make sure to have a look at what it contains in the text editor or by inspecting its members). 

In [73]:
import message as signed_msg

Import my public certificate as well (same comments apply):

In [74]:
from certs import CHENEVERT_Gabriel as cert

1 - Check that the parameters $(p,q,g)$ given in the certificate form a valid triple for DSA.

In [75]:

from is_prime import is_prime

def q_divides_p_minus_1(p,q,g):
    return (p - 1) % q == 0

def g_exp_q_mod_p_equals_1(p,q,g):
    return pow(g, q, p) == 1

def g_is_in_range(p,q,g):
    return g > 1 and g < p

def check_dsa_params(p, q, g):
    checks = [ q_divides_p_minus_1, g_exp_q_mod_p_equals_1, g_is_in_range ]
    return is_prime(p) and is_prime(q) and all([check(p,q,g) for check in checks])



In [76]:
p = cert.p
q = cert.q
g = cert.g

check_dsa_params(p,q,g)

True

2 - Then verify that the message signature checks out correctly. The hash function that was used is simply to take the number of characters in the message (mod $q$).

In [102]:
import hashlib

def hash_msg(msg, algo):
    if algo == "DSA":
        H = len(msg)%q
    elif algo == "SHA256-DSA":
        h_bytes = hashlib.sha256(msg.encode("utf-8")).digest()
        H = int.from_bytes(h_bytes, "big") % q
    return H

def verify_signature(msg, r, s, p, algo, q, g, y):
    H = hash_msg(msg, algo)
    w = pow(s, -1, q)
    u1 = (H * w) % q
    u2 = (r * w) % q
    v = (pow(g, u1, p) * pow(y, u2, p)) % p % q
    return v == r

def verify_msg_cert(msg, cert):
    return verify_signature(msg.m, msg.r, msg.s, cert.p, msg.algorithm, cert.q, cert.g, cert.y)


In [78]:
print("Signature valide:", verify_msg_cert(signed_msg, cert))

Signature valide: True


3 - This is, however, a <i>very</i> insecure hash function. Can you see how an attacker could use this to forge a message bearing my signature? (the signature has to appear valid to anyone that verifies it).

Avec un message forgé de longueur : len(msg original) + k * q, la signature du message forgé sera identique à celle du message initial.


In [79]:
def copy_msg(msg):
    from types import SimpleNamespace

    fields = ["m", "r", "s", "issuer", "algorithm"]
    data = {f: getattr(msg, f, None) for f in fields}  # None si absent
    return SimpleNamespace(**data)

In [81]:
l = len(signed_msg.m)
forged_message = copy_msg(signed_msg)
forged_message.m =  "@" * l
print(forged_message.m)
print("Signature valide:", verify_msg_cert(forged_message, cert))

@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
Signature valide: True


## B) Verification, pt. 2

By now, your favorite teacher should have posted a message on the public, unauthenticated shared text file [found here](https://myjunia-my.sharepoint.com/:t:/g/personal/gabriel_chenevert_junia_com/EfIrvlehK0FLj1ESG6OmUygBtJrQNZYOU-xxHoYVoE_TrQ?e=wfuUV7). Verify that this signature is legit using the public key in my certificate.

In [91]:
import shared_message
print("Signature valide:", verify_msg_cert(shared_message, cert))

import shared_message_2
from certs import GOLDBLUM_Jeff as cert_jeff
print("Signature valide:", verify_msg_cert(shared_message_2, cert_jeff))


Signature valide: True
Signature valide: True


Actually: the above verification only guarantees that the person who signed this message had access to the private key corresponding to the public key in the certificate. To bind this public key to a given identity, _the certificate itself should be checked_. Which means: verify that the CA signature on my certificate is legit. The message here is the string corresponding to the PUBLIC KEY (including commented lines).

In [96]:
def extract_pk(cert):
    import inspect
    source_text = inspect.getsource(cert)
    pubkey_block = source_text.split("# BEGIN PUBLIC KEY")[1].split("# END PUBLIC KEY")[0]
    pubkey_block = "# BEGIN PUBLIC KEY" + pubkey_block + """# END PUBLIC KEY
"""
    return pubkey_block

In [99]:
def check_cert(cert, issuer_cert):
    cert_msg = copy_msg(cert)
    cert_msg.m = extract_pk(cert)
    return verify_msg_cert(cert_msg, issuer_cert)
    
    


In [ ]:
from certs import BIDON_Charles_Antoine as certif_ca
print("Certificat valide:", check_cert(cert, certif_ca))

Certificat valide: True


Should something else be verified in order to have a full chain of trust?

Il faut s'assurer que le certificat de Charles Antoine BIDON est digne de confiance.

In [101]:
print("Certificat valide:", check_cert(certif_ca, certif_ca))

Certificat valide: True


## C) Sign message

Your turn now: you should have received your private key from the CA. Check that the $(x,y)$ it contains is indeed a valid key pair for DSA with the given $(p,q,g)$.

In [ ]:
import random

def sign_message(msg, cert, x, algo="SHA256-DSA"):
    q = cert.q
    p = cert.p
    g = cert.g
    H = hash_msg(msg, q, algo)
    
    while True:
        k = random.randrange(1, q)  # k aléatoire secret
        r = pow(g, k, p) % q
        if r == 0:
            continue
        k_inv = pow(k, -1, q)
        s = (k_inv * (H + x * r)) % q
        if s != 0:
            break
    return r, s

def signed_message(msg, name, r, s, algo="SHA256-DSA"):
    return f"""# BEGIN SIGNED MESSAGE

m = "{msg}"

# BEGIN SIGNATURE

issuer = "{name}"

algorithm = "{algo}"

r = {hex(r)}

s = {hex(s)}

# END SIGNATURE

# END SIGNED MESSAGE
"""


In [ ]:
x = 0x9d679557c3086e045328f2952a09191cb37319bec352cc4a71513276852d87d3

from certs import HERSSENS_Alexandre as my_cert






Then compute the SHA256-DSA signature of a message of your choice with this private key. You may post the signed message on the public message board above so that anyone may verify your signature.

## D) Importance of CA

Being a certificate authority is not a role that should be taken lightly. Indeed, if you take a look at the certificates generated by Charles-Antoine, you'll see that there is a serious problem with his implementation of DSA: all components $r$ of his signatures are equal! What mistake could have been committed?

Exploit this mistake by recovering the CA private key, hence gaining the ability to generate your own ("verifiable") certificates. Test this ability by forging a fake one! (<i>i.e.</i>, a certificate not issued by Charles-Antoine but apparently signed by him, that anyone would accept as such) 